# Step 1A: Data Loading and Preprocessing

## Overview
This notebook loads ROSMAP (Religious Orders Study and Memory and Aging Project) proteomics data and clinical metadata, applies quality control (QC), normalization, and covariate adjustment to prepare clean data for downstream analysis.

## Scientific Rationale
Bulk proteomics data is subject to multiple sources of technical and biological variation:
- **Missing values** from detection limits or sample failures → QC filtering and KNN imputation
- **Batch effects and normalization** from instrument drift → log2 transformation + Z-score normalization
- **Technical covariates** (age at death, sex, post-mortem interval) → Linear regression residuals

This step produces a clean, normalized proteomics matrix aligned with clinical metadata.

## Key Inputs
- `data/raw/proteomics_matrix.csv`: Bulk proteomics (patients × proteins)
- `data/raw/clinical_metadata.csv`: Clinical phenotypes (diagnosis, Braak stage, MMSE, CERAD)

## Expected Outputs
- `data/processed/rosmap_proteomics_cleaned.csv`: Cleaned proteomics matrix (n_samples × n_proteins)
- `data/processed/rosmap_metadata.csv`: Aligned clinical metadata
- `results/step1/qc_report.png`: 2-panel QC figure (missing value histogram + PCA plot)

## Execution Time
- **Full data**: ~5-10 minutes (depends on matrix size)
- **Test mode**: ~2 seconds

## Success Criteria
- ✓ All missing values imputed (0 NaN values after step 5)
- ✓ Proteomics normalized (Z-score: mean ≈ 0, std ≈ 1)
- ✓ Samples aligned between proteomics and metadata (n_common > 150)
- ✓ QC report generated without errors

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.facecolor'] = 'white'

### Library Overview
- **pandas/numpy**: Data manipulation and numerical operations
- **matplotlib/seaborn**: Visualization
- **sklearn**: Machine learning tools (KNN imputation, PCA, standard scaler, linear regression)
- **scipy**: Statistical functions and optimization
- **warnings**: Suppress non-critical messages

## Configuration

In [ ]:
# Data paths
RAW_DATA_DIR = "../data/raw"
PROCESSED_DATA_DIR = "../data/processed"
RESULTS_DIR = "../results/step1"

# Parameters
MISSING_VALUE_THRESHOLD = 0.50  # Remove proteins with >50% missing values
KNN_K = 5  # Number of neighbors for imputation
LOG2_PSEUDOCOUNT = 1  # Pseudocount for log2 transformation
PCA_N_COMPONENTS = 2  # For visualization

### Data Preprocessing Pipeline

The following steps transform raw proteomics into a clean, normalized, analysis-ready matrix:

1. **Load & Join** (Steps 1-3): Combine proteomics and clinical metadata on patient IDs
2. **QC Filtering** (Step 4): Remove proteins with >50% missing values (unreliable measurements)
3. **KNN Imputation** (Step 5): Fill remaining missing values using k-nearest neighbors (non-parametric)
4. **Log2 Transform** (Step 6): Stabilize variance (proteomics are log-normally distributed)
5. **Z-score Normalization** (Step 7): Center/scale each protein across all samples (mean=0, std=1)
6. **Covariate Regression** (Step 8): Remove technical effects (age, sex, PMI) via linear regression residuals

Each step is tracked with detailed logging to catch data quality issues early.

def load_proteomics_data(file_path):
    """
    Load proteomics matrix and auto-detect orientation.
    
    Why this matters: Raw proteomics data may come in either orientation
    (genes × samples or samples × genes). This function handles both.
    
    Returns matrix with shape (n_samples x n_proteins).
    """
    print("[1] Loading proteomics data...")
    df = pd.read_csv(file_path, index_col=0)

    # Auto-detect orientation: if more columns than rows, likely transposed
    n_rows, n_cols = df.shape
    print(f"  Initial shape: {n_rows} rows x {n_cols} columns")

    # Heuristic: ROSMAP has ~5,000+ proteins, so if rows >> cols, transpose is needed
    # This prevents the common error of accidentally using (n_proteins × n_samples) orientation
    if n_rows < n_cols and n_cols > 1000:
        print("  Auto-detected transposed matrix, transposing...")
        df = df.T
        print(f"  After transpose: {df.shape}")

    print(f"  Final proteomics shape: {df.shape[0]} samples x {df.shape[1]} proteins")
    return df

In [ ]:
def load_proteomics_data(file_path):
    """
    Load proteomics matrix and auto-detect orientation.
    Returns matrix with shape (n_samples x n_proteins).
    """
    print("[1] Loading proteomics data...")
    df = pd.read_csv(file_path, index_col=0)

    # Auto-detect orientation: if more columns than rows, likely transposed
    n_rows, n_cols = df.shape
    print(f"  Initial shape: {n_rows} rows x {n_cols} columns")

    # Heuristic: ROSMAP has ~5,000+ proteins, so if rows >> cols, transpose is needed
    if n_rows < n_cols and n_cols > 1000:
        print("  Auto-detected transposed matrix, transposing...")
        df = df.T
        print(f"  After transpose: {df.shape}")

    print(f"  Final proteomics shape: {df.shape[0]} samples x {df.shape[1]} proteins")
    return df

In [ ]:
def load_clinical_metadata(file_path):
    """
    Load clinical metadata. Handles common column name variants.
    """
    print("[2] Loading clinical metadata...")
    df = pd.read_csv(file_path, index_col=False)
    print(f"  Metadata shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()[:10]}...")  # Preview first 10 columns
    return df

In [ ]:
def find_patient_id_column(metadata_df):
    """
    Find the patient ID column in metadata (handles common variants).
    """
    patient_id_cols = ['projid', 'project_id', 'proj_id', 'patient_id', 'PROJID']
    for col in patient_id_cols:
        if col in metadata_df.columns:
            print(f"  Found patient ID column: '{col}'")
            return col

    raise ValueError(f"Could not find patient ID column. Columns: {metadata_df.columns.tolist()}")

In [ ]:
def join_proteomics_and_metadata(proteomics_df, metadata_df):
    """
    Join proteomics matrix with clinical metadata on patient ID.
    """
    print("[3] Joining proteomics and metadata...")

    # Find patient ID column
    patient_id_col = find_patient_id_column(metadata_df)

    # Set patient ID as index in metadata
    metadata_indexed = metadata_df.set_index(patient_id_col)

    # Find common samples between proteomics and metadata
    common_samples = proteomics_df.index.intersection(metadata_indexed.index)
    print(f"  Samples in proteomics: {len(proteomics_df)}")
    print(f"  Samples in metadata: {len(metadata_indexed)}")
    print(f"  Common samples: {len(common_samples)}")

    # Subset both to common samples
    proteomics_df = proteomics_df.loc[common_samples]
    metadata_df = metadata_indexed.loc[common_samples]

    return proteomics_df, metadata_df

In [ ]:
def knn_impute_missing_values(proteomics_df, k=5):
    """
    Impute remaining missing values using KNN imputation.
    
    Why KNN?: It leverages similarity between samples without assuming
    normal distribution. The 'distance' weighting means closer neighbors
    contribute more to the imputed value.
    
    Parameter choice (k=5): Balance between stability (high k) and
    local adaptation (low k). k=5 is standard for biological datasets.
    """
    print("[5] KNN imputation (k={})...".format(k))

    n_missing_before = proteomics_df.isnull().sum().sum()
    print(f"  Missing values before imputation: {n_missing_before}")

    # Initialize KNN imputer with distance weighting
    # This means the 5 nearest neighbors contribute proportionally to their distance
    imputer = KNNImputer(n_neighbors=k, weights='distance')
    
    # Fit and transform the data
    proteomics_array = imputer.fit_transform(proteomics_df)
    
    # Convert back to DataFrame with original indices/columns
    proteomics_df = pd.DataFrame(
        proteomics_array,
        columns=proteomics_df.columns,
        index=proteomics_df.index
    )

    n_missing_after = proteomics_df.isnull().sum().sum()
    print(f"  Missing values after imputation: {n_missing_after}")

    return proteomics_df

In [ ]:
def qc_filter_proteins(proteomics_df, threshold=0.50):
    """
    Remove proteins with >threshold% missing values.
    """
    print("[4] Quality control filtering...")

    missing_pct = (proteomics_df.isnull().sum() / len(proteomics_df) * 100)
    proteins_before = len(proteomics_df.columns)

    # Keep proteins below threshold
    proteins_to_keep = missing_pct[missing_pct <= (threshold * 100)].index
    proteomics_df = proteomics_df[proteins_to_keep]

    proteins_after = len(proteomics_df.columns)
    removed = proteins_before - proteins_after
    print(f"  Proteins before QC: {proteins_before}")
    print(f"  Proteins removed (>50% missing): {removed}")
    print(f"  Proteins after QC: {proteins_after}")

    return proteomics_df, missing_pct

In [ ]:
def knn_impute_missing_values(proteomics_df, k=5):
    """
    Impute remaining missing values using KNN imputation.
    """
    print("[5] KNN imputation (k={})...".format(k))

    n_missing_before = proteomics_df.isnull().sum().sum()
    print(f"  Missing values before imputation: {n_missing_before}")

    imputer = KNNImputer(n_neighbors=k, weights='distance')
    proteomics_array = imputer.fit_transform(proteomics_df)
    proteomics_df = pd.DataFrame(
        proteomics_array,
        columns=proteomics_df.columns,
        index=proteomics_df.index
    )

    n_missing_after = proteomics_df.isnull().sum().sum()
    print(f"  Missing values after imputation: {n_missing_after}")

    return proteomics_df

In [ ]:
def regress_out_covariates(proteomics_df, metadata_df, covariate_cols=['age_death', 'msex', 'pmi']):
    """
    Regress out technical covariates using linear regression.
    Returns residuals as the cleaned expression matrix.
    
    Mathematical approach:
      protein_i = β₀ + β₁*age + β₂*sex + β₃*pmi + residuals
      We keep only the residuals (unexplained variance)
    
    This removes systematic effects of age, sex, and PMI while preserving
    biologically meaningful protein-protein correlations.
    """
    print("[8] Removing technical covariates...")

    # Find available covariate columns (handle naming variants)
    covariates_available = {}
    for col in covariate_cols:
        # Try exact match and common variants
        variants = [col, col.lower(), col.upper()]
        for var in variants:
            if var in metadata_df.columns:
                covariates_available[col] = var
                break

    print(f"  Available covariates: {list(covariates_available.keys())}")

    # Prepare covariate matrix
    X = metadata_df[[covariates_available[key] for key in covariates_available]].copy()

    # Handle categorical variables (e.g., sex: typically 0=female, 1=male)
    for col in X.columns:
        if X[col].dtype == 'object':
            # Convert categorical to numeric codes
            X[col] = pd.Categorical(X[col]).codes

    # Handle any remaining missing values in covariates (fill with mean)
    X = X.fillna(X.mean())

    # Fit linear model and get residuals for each protein
    # We do this protein-by-protein to preserve correlations between proteins
    proteomics_residuals = proteomics_df.copy()
    for protein in proteomics_df.columns:
        y = proteomics_df[protein].values
        
        # Fit linear model: protein ~ covariates
        model = LinearRegression()
        model.fit(X, y)
        
        # Get residuals (actual - predicted)
        residuals = y - model.predict(X)
        proteomics_residuals[protein] = residuals

    print(f"  Regression applied to {proteomics_df.shape[1]} proteins")

    return proteomics_residuals

### Covariate Adjustment Strategy

Technical covariates (age at death, sex, post-mortem interval) can confound biological signals. Linear regression residuals remove these effects while preserving protein-protein correlations. We regress each protein independently, keeping the residuals (unexplained variance) for downstream analysis.

In [ ]:
def zscore_normalize(proteomics_df):
    """
    Z-score normalize each protein (across all samples).
    """
    print("[7] Z-score normalization...")

    scaler = StandardScaler()
    proteomics_array = scaler.fit_transform(proteomics_df)
    proteomics_df = pd.DataFrame(
        proteomics_array,
        columns=proteomics_df.columns,
        index=proteomics_df.index
    )

    # Verify normalization
    means = proteomics_df.mean(axis=0)
    stds = proteomics_df.std(axis=0)
    print(f"  Mean after Z-score: mean={means.mean():.2e}, std={stds.mean():.4f}")

    return proteomics_df

In [ ]:
## Main Pipeline Execution

The following cell orchestrates all preprocessing steps in order. Progress is logged at each stage with key metrics (n_samples, n_proteins, missing values, normalization statistics).

**Expected output**:
- Console logs showing progress through each step
- Final summary with output file locations
- Ready signal for Step 1B (Deconvolution)

In [ ]:
def save_processed_data(proteomics_df, metadata_df, processed_dir):
    """
    Save cleaned proteomics matrix and metadata.
    """
    print("[9] Saving processed data...")

    proteomics_file = f"{processed_dir}/rosmap_proteomics_cleaned.csv"
    metadata_file = f"{processed_dir}/rosmap_metadata.csv"

    proteomics_df.to_csv(proteomics_file)
    metadata_df.to_csv(metadata_file)

    print(f"  Saved: {proteomics_file}")
    print(f"  Saved: {metadata_file}")

In [ ]:
def generate_qc_report(proteomics_df_before_qc, proteomics_df_after,
                       metadata_df, missing_pct_before, results_dir):
    """
    Generate QC report figure with:
    1. Histogram of missing values per protein (before QC)
    2. PCA plot colored by diagnosis (after preprocessing)
    """
    print("[10] Generating QC report figure...")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel 1: Histogram of missing values before QC
    ax = axes[0]
    ax.hist(missing_pct_before, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(50, color='red', linestyle='--', linewidth=2, label='50% threshold')
    ax.set_xlabel('Missing Value Percentage (%)', fontsize=11)
    ax.set_ylabel('Number of Proteins', fontsize=11)
    ax.set_title('Missing Values per Protein (Before QC)', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    # Panel 2: PCA plot colored by diagnosis
    ax = axes[1]

    # Apply PCA
    pca = PCA(n_components=PCA_N_COMPONENTS)
    pca_coords = pca.fit_transform(proteomics_df_after)

    # Find diagnosis column
    diagnosis_col = None
    for col in ['diagnosis', 'cogdx', 'Diagnosis']:
        if col in metadata_df.columns:
            diagnosis_col = col
            break

    if diagnosis_col:
        diagnosis = metadata_df[diagnosis_col].values
        unique_diagnoses = np.unique(diagnosis)

        # Map diagnoses to colors
        colors = plt.cm.Set2(np.linspace(0, 1, len(unique_diagnoses)))
        color_map = {diag: colors[i] for i, diag in enumerate(unique_diagnoses)}

        for diag in unique_diagnoses:
            mask = diagnosis == diag
            ax.scatter(pca_coords[mask, 0], pca_coords[mask, 1],
                      label=str(diag), alpha=0.6, s=50, color=color_map[diag])

        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
        ax.set_title('PCA of Samples (After Preprocessing)', fontsize=12, fontweight='bold')
        ax.legend(title='Diagnosis', fontsize=9)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    qc_report_file = f"{results_dir}/qc_report.png"
    plt.savefig(qc_report_file, dpi=300, bbox_inches='tight')
    print(f"  Saved: {qc_report_file}")
    plt.close()

## Main Pipeline

In [ ]:
print("\n" + "="*70)
print("STEP 1A: DATA LOADING & PREPROCESSING")
print("="*70 + "\n")

try:
    # Load data
    proteomics_df = load_proteomics_data(f"{RAW_DATA_DIR}/proteomics_matrix.csv")
    metadata_df = load_clinical_metadata(f"{RAW_DATA_DIR}/clinical_metadata.csv")

    # Save original for QC comparison
    proteomics_df_original = proteomics_df.copy()

    # Join data
    proteomics_df, metadata_df = join_proteomics_and_metadata(proteomics_df, metadata_df)

    # Print initial summary
    missing_pct_before_qc = print_summary_statistics(proteomics_df, metadata_df)

    # QC filtering
    proteomics_df, missing_pct_before = qc_filter_proteins(proteomics_df, threshold=MISSING_VALUE_THRESHOLD)

    # Imputation
    proteomics_df = knn_impute_missing_values(proteomics_df, k=KNN_K)

    # Log2 transformation
    proteomics_df = log2_transform(proteomics_df, pseudocount=LOG2_PSEUDOCOUNT)

    # Z-score normalization
    proteomics_df = zscore_normalize(proteomics_df)

    # Covariate adjustment
    proteomics_df = regress_out_covariates(proteomics_df, metadata_df)

    # Save processed data
    save_processed_data(proteomics_df, metadata_df, PROCESSED_DATA_DIR)

    # Generate QC report (before QC comparison)
    generate_qc_report(proteomics_df_original, proteomics_df,
                      metadata_df, missing_pct_before_qc, RESULTS_DIR)

    print("\n" + "="*70)
    print("PREPROCESSING COMPLETE ✓")
    print("="*70)
    print(f"\nProcessed data saved to: {PROCESSED_DATA_DIR}")
    print(f"QC report saved to: {RESULTS_DIR}")
    print("\nReady for Step 1B: Deconvolution & Pseudotime Analysis\n")

except Exception as e:
    print(f"\n❌ ERROR: {str(e)}")
    import traceback
    traceback.print_exc()